## Standard - Same Equity Curve

**2025-06-25** — All sim / live / notebook parity edits are tagged with `Standard - Same Equity Curve`. Search the repo for that string to list every touchpoint.

### What this standard covers
- Shared features: `features.py` → `FEATURE_COLS` + `add_features()`
- Production joblib: `Models/BTCUSDT4h1307.joblib` (set `MODEL_PATH` in `.env`)
- Equity-curve alignment: same model inputs as `export_simulation_equity_curve.py` + `api_liveScript.py`

### Still required for identical equity curves (see `features.py`)
1. Retrain with this notebook after the ADX / feature unification
2. Align `.env`: `THRESHOLD=0.4214`, `TAKE_PROFIT=0.0455`, `STOP_LOSS=0.0051`, `PCT_ACCOUNT_PER_TRADE=0.7888`
3. Use the same joblib in live bot and export script
4. Execution gaps remain (market fill vs close, OCO vs close-only TP/SL)


In [1]:
# Cell 1 - Imports & basic config

import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)

# Path to OHLCV CSV. Native bar size must match BASE_TIMEFRAME (or leave BASE_TIMEFRAME
# as a label and rely on BASE_BAR_MINUTES from the load cell).
# For 1h/15m resample search, use 1m base data (see export_simulation_equity_curve.py).
DATA_PATH = "BINANCE_BTCUSDT_4H_20May.csv"
BASE_TIMEFRAME = "4h"  # native bar size in the CSV

# Global train/test period (will be reused across all configs)
TRAIN_START_DATE = None          # None = from very beginning
TRAIN_END_DATE   = "2021-11-08"

TEST_START_DATE  = "2021-11-09"
TEST_END_DATE    = None          # None = until very end

INITIAL_EQUITY = 100_000.0
COMMISSION_PCT = 0.001           # 0.1% per side

# Random search reproducibility (see generate_search_trials in resample cell).
# Same SEARCH_SEED + N_TRIALS + RESAMPLE_OPTIONS + DATA_PATH -> identical trial configs.
SEARCH_SEED = 42
N_TRIALS = 800
XGBOOST_RANDOM_STATE = 42        # default model seed when evaluate_config random_state omitted

# Default simulation exit policy for evaluate_config (overridable per call).
# "tp_sl" = only TP/SL (+ EOD flatten). "horizon" = exit after horizon_steps bars at close.
# "tp_sl_or_horizon" = TP/SL, else time exit after horizon_steps bars.

EXIT_MODE = "tp_sl_or_horizon" # "tp_sl" | "horizon" | "tp_sl_or_horizon"

"""
    Run a trading simulation on the test period using predicted buy signals.

    exit_mode
      * "tp_sl": exit only via take-profit / stop-loss (or forced flatten at end of test data).
      * "horizon": exit at the close of the bar when the trade has been open for `horizon_steps`
        bars (no TP/SL). Any position still open at the last bar closes at "EOD".
      * "tp_sl_or_horizon": TP/SL as above; if neither triggers within `horizon_steps` bars after
        entry, exit at that bar's close with reason "HORIZON".

    Daily CVaR stop (when cvar_loss_limit_pct_of_cvar is not None):
      Rolling CVaR of *completed* calendar-day returns on the strategy equity curve.
      If today's loss (from today's open equity to now) exceeds
          (cvar_loss_limit_pct_of_cvar / 100) * |CVaR|,
      set risk halt: no new trades; optionally flatten all positions at this bar's close ("CVAR_HALT").
"""


# Daily CVaR circuit breaker (run_simulation). None = disabled.
# Halt if calendar-day loss (vs day open equity) exceeds (X/100) * |rolling CVaR|.
CVAR_LOSS_LIMIT_PCT_OF_CVAR = None  # e.g. 100.0
CVAR_ALPHA = 0.05
CVAR_LOOKBACK_DAYS = 60
CVAR_MIN_HISTORY_DAYS = 20
CVAR_HALT_FLATTEN = True

import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Standard - Same Equity Curve (2025-06-25)
# Imports shared feature pipeline used by export_simulation_equity_curve.py
# and api_liveScript.py so training matches live inference.
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

_REPO_ROOT = Path(".").resolve()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from features import FEATURE_COLS, add_features

# Standard - Same Equity Curve (2025-06-25): production joblib for live bot — set MODEL_PATH in .env after cell 18
MODEL_OUTPUT_PATH = "Models/BTCUSDT4h1307.joblib"


## load data

In [2]:
# Supports:
#   - Binance export: Date, Time, Open, High, Low, Close, Volume, Source (UTC-6 in file -> converted to UTC)
#   - TradingView export: C (unix seconds), open/high/low/close, optional open interest
df_raw = pd.read_csv(DATA_PATH, comment='#')
df_raw.columns = df_raw.columns.str.strip()

if 'Date' in df_raw.columns and 'Time' in df_raw.columns:
    df_raw['DateTime'] = pd.to_datetime(
        df_raw['Date'].astype(str).str.zfill(8) + ' ' + df_raw['Time'].astype(str),
        format='%Y%m%d %H:%M:%S',
        errors='coerce',
    )
    # File header notes Date/Time are UTC-6; Binance klines are UTC
    df_raw['DateTime'] = df_raw['DateTime'] + pd.Timedelta(hours=6)
else:
    rename_map = {}
    for col in df_raw.columns:
        key = col.lower()
        if col == 'C' or key in ('time', 'timestamp', 'datetime'):
            rename_map[col] = 'DateTime'
        elif key == 'open':
            rename_map[col] = 'Open'
        elif key == 'high':
            rename_map[col] = 'High'
        elif key == 'low':
            rename_map[col] = 'Low'
        elif key == 'close':
            rename_map[col] = 'Close'
        elif key in ('volume', 'vol'):
            rename_map[col] = 'Volume'
    df_raw = df_raw.rename(columns=rename_map)
    if 'DateTime' not in df_raw.columns:
        raise KeyError(f"No timestamp column found. Columns: {list(df_raw.columns)}")
    df_raw['DateTime'] = pd.to_datetime(df_raw['DateTime'], unit='s', utc=True).dt.tz_localize(None)

# Normalize OHLCV names (Binance export already uses Open/High/Low/Close/Volume)
col_aliases = {
    'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close',
    'volume': 'Volume', 'vol': 'Volume',
}
df = df_raw.rename(columns={c: col_aliases[c.lower()] for c in df_raw.columns if c.lower() in col_aliases})

if 'Volume' not in df.columns:
    oi_close_col = next(
        (c for c in df_raw.columns if 'open interest candles (close' in c.lower()),
        None,
    )
    if oi_close_col:
        df['Volume'] = pd.to_numeric(df_raw[oi_close_col], errors='coerce')
        print(f"Mapped '{oi_close_col}' -> 'Volume' (open interest proxy)")
    else:
        df['Volume'] = 0.0
        print('No volume column found; Volume set to 0')

ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in ohlcv_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df_min = (
    df[['DateTime'] + ohlcv_cols]
    .dropna(subset=['Open', 'High', 'Low', 'Close'])
    .sort_values('DateTime')
    .drop_duplicates(subset='DateTime', keep='last')
    .set_index('DateTime')
)

# Median spacing of loaded bars — used to forbid resample rules finer than the source CSV.
_bar_spacing = df_min.index.to_series().diff().dropna()
BASE_BAR_MINUTES = float(_bar_spacing.median().total_seconds() / 60.0)

print(f'Loaded {len(df_min):,} {BASE_TIMEFRAME} bars: {df_min.index.min()} -> {df_min.index.max()}')
print(f'Native bar spacing: {BASE_BAR_MINUTES:.0f} min ({BASE_BAR_MINUTES / 60:.1f} h)')
print('Columns kept for pipeline:', list(df_min.columns))
df_min.head()

Loaded 19,177 4h bars: 2017-08-17 04:00:00 -> 2026-05-20 20:00:00
Columns kept for pipeline: ['Open', 'High', 'Low', 'Close', 'Volume']


,Open,High,Low,Close,Volume
DateTime,,,,,
2017-08-17 04:00:00,4261.48,4349.99,4261.32,4349.99,82
2017-08-17 08:00:00,4333.32,4485.39,4333.32,4427.30,63
2017-08-17 12:00:00,4436.06,4485.39,4333.42,4352.34,174
2017-08-17 16:00:00,4352.33,4354.84,4200.74,4325.23,225
2017-08-17 20:00:00,4307.56,4369.69,4258.56,4285.08,249


## resample

In [3]:
# Cell 3 - Build dataset for given RESAMPLE_RULE, HORIZON_STEPS, TARGET_SIMPLE_RETURN
#
# Standard - Same Equity Curve (2025-06-25):
#   - Indicators: features.add_features() (Wilder ADX, same as live bot)
#   - Removed SMA51W_diff_pct (not in live FEATURE_COLS)
#   - Labels only: future_ret / target_buy added here after shared features
#
# Resampling rule: only aggregate to SAME or COARSER bars than the source CSV.
# Downsampling (e.g. 4h -> 1h) creates empty hourly buckets; dropna() removes them
# and you keep the same sparse series — no real 1h OHLCV, and labels/sim exits lie
# about bar size. For multi-timeframe search, load 1m/1h base data (see export script).

_OHLCV_AGG = {
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
}


def _rule_minutes(resample_rule: str) -> float:
    return pd.Timedelta(resample_rule.replace("min", "m")).total_seconds() / 60.0


def resample_bars(df_base: pd.DataFrame, resample_rule: str) -> pd.DataFrame:
    """Aggregate OHLCV to resample_rule. Raises if rule is finer than loaded bar spacing."""
    target_min = _rule_minutes(resample_rule)
    if target_min < BASE_BAR_MINUTES * 0.99:
        raise ValueError(
            f"Cannot resample {BASE_BAR_MINUTES:.0f}min source bars to '{resample_rule}' "
            f"({target_min:.0f}min): empty buckets are dropped — no finer OHLCV exists. "
            "Use coarser resample_rule or load finer base data (e.g. 1m CSV)."
        )

    agg = dict(_OHLCV_AGG)
    if "Volume" in df_base.columns:
        agg["Volume"] = "sum"

    df_bars = df_base.resample(resample_rule).agg(agg).dropna()
    if df_bars.empty:
        raise RuntimeError(f"No bars after resampling to {resample_rule!r}.")
    return df_bars


def resample_options_for_base(base_bar_minutes: float) -> dict[str, list[int]]:
    """Horizon grids per allowed resample rule (same or coarser than source only)."""
    candidates = {
        "15min": [32, 64, 96, 144, 192],
        "1h": [1, 8, 24, 48],
        # 24/48 on 4h ≈ old fake '1h'/24 and '1h'/48 wall-clock holds
        "4h": [1, 4, 6, 8, 12, 24, 48],
        "1D": [1, 2, 3, 7],
    }
    return {
        rule: horizons
        for rule, horizons in candidates.items()
        if _rule_minutes(rule) >= base_bar_minutes * 0.99
    }


def search_grid_fingerprint(resample_options: dict[str, list[int]]) -> str:
    """Short hash of the horizon grid — changes when RESAMPLE_OPTIONS content changes."""
    import hashlib
    import json

    payload = json.dumps({k: list(v) for k, v in sorted(resample_options.items())}, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()[:8]


def generate_search_trials(
    n_trials: int,
    search_seed: int,
    resample_options: dict[str, list[int]],
) -> list[dict]:
    """
    Build the full random-search trial list from ONE seed.

    Deterministic when n_trials, search_seed, and resample_options are unchanged.
    Resample rules are sorted before rng.choice so dict insertion order never matters.
    """
    rng = np.random.default_rng(int(search_seed))
    rules = sorted(resample_options.keys())
    horizons_by_rule = {k: list(resample_options[k]) for k in rules}

    trials = []
    for trial_idx in range(int(n_trials)):
        resample_rule = str(rng.choice(rules))
        horizon_steps = int(rng.choice(horizons_by_rule[resample_rule]))
        trials.append(
            {
                "trial": trial_idx,
                "search_seed": int(search_seed),
                "grid_fingerprint": search_grid_fingerprint(resample_options),
                "resample_rule": resample_rule,
                "horizon_steps": horizon_steps,
                "target_simple_return": float(rng.uniform(0.003, 0.15)),
                "take_profit_pct": float(rng.uniform(0.01, 0.10)),
                "stop_loss_pct": float(rng.uniform(0.005, 0.05)),
                "max_open_trades": int(rng.integers(1, 11)),
                "pct_account_per_trade": float(rng.uniform(0.05, 0.50)),
                "threshold": float(rng.uniform(0.3, 0.9)),
                "model_random_state": int(search_seed) + trial_idx,
            }
        )
    return trials


def replay_search_trial(
    trial_idx: int,
    search_seed: int = SEARCH_SEED,
    resample_options: dict[str, list[int]] | None = None,
    **evaluate_kwargs,
):
    """Re-run one search trial by index (0-based) with the same config + model seed."""
    opts = resample_options if resample_options is not None else resample_options_for_base(BASE_BAR_MINUTES)
    trials = generate_search_trials(trial_idx + 1, search_seed, opts)
    t = trials[trial_idx]
    return evaluate_config(
        resample_rule=t["resample_rule"],
        horizon_steps=t["horizon_steps"],
        target_simple_return=t["target_simple_return"],
        take_profit_pct=t["take_profit_pct"],
        stop_loss_pct=t["stop_loss_pct"],
        max_open_trades=t["max_open_trades"],
        pct_account_per_trade=t["pct_account_per_trade"],
        threshold=t["threshold"],
        random_state=t["model_random_state"],
        verbose=True,
        **evaluate_kwargs,
    )


def build_dataset(resample_rule, horizon_steps, target_simple_return):
    """
    From global df_min, resample bars, compute shared features, then training labels.

    Standard - Same Equity Curve (2025-06-25): df_tf uses features.py; df_model drops rows without future labels.
    """
    df_bars = resample_bars(df_min, resample_rule)
    # Standard - Same Equity Curve (2025-06-25): single feature path for notebook, export script, and live API
    df_tf = add_features(df_bars, horizon_steps=horizon_steps)

    future_close = df_tf["Close"].shift(-horizon_steps)
    df_tf["future_ret"] = (future_close - df_tf["Close"]) / df_tf["Close"]
    df_tf["target_buy"] = (df_tf["future_ret"] >= target_simple_return).astype(int)

    df_model = df_tf.dropna(subset=["future_ret"]).copy()
    return df_tf, df_model


## TRAIN TEST SPLIT


In [4]:
# Cell 4 - Train/test split by date

def split_train_test(df_model, feature_cols,
                     train_start=TRAIN_START_DATE,
                     train_end=TRAIN_END_DATE,
                     test_start=TEST_START_DATE,
                     test_end=TEST_END_DATE):
    idx = df_model.index

    train_start = pd.to_datetime(train_start) if train_start is not None else idx.min()
    train_end   = pd.to_datetime(train_end)   if train_end   is not None else idx.max()
    test_start  = pd.to_datetime(test_start)  if test_start  is not None else idx.min()
    test_end    = pd.to_datetime(test_end)    if test_end    is not None else idx.max()

    # Ensure timestamps share the same tz-awareness as the index to avoid comparison errors
    idx_tz = getattr(idx, 'tz', None)
    def _localize_to_index_tz(ts, tz):
        if pd.isna(ts):
            return ts
        ts = pd.to_datetime(ts)
        if tz is None:
            return ts
        if getattr(ts, 'tz', None) is None:
            try:
                return ts.tz_localize(tz)
            except Exception:
                return ts
        return ts.tz_convert(tz)

    train_start = _localize_to_index_tz(train_start, idx_tz)
    train_end   = _localize_to_index_tz(train_end, idx_tz)
    test_start  = _localize_to_index_tz(test_start, idx_tz)
    test_end    = _localize_to_index_tz(test_end, idx_tz)

    train_mask = (idx >= train_start) & (idx <= train_end)
    test_mask  = (idx >= test_start)  & (idx <= test_end)

    X = df_model[feature_cols]
    y = df_model['target_buy']

    X_train, y_train = X.loc[train_mask], y.loc[train_mask]
    X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

    return X_train, X_test, y_train, y_test, train_start, train_end, test_start, test_end

## TRAIN XGBOOST

In [5]:
# Cell 5 - Train standard XGBoost classifier

def train_classifier(X_train, y_train, random_state=XGBOOST_RANDOM_STATE):
    # Class imbalance weight
    pos = y_train.sum()
    neg = len(y_train) - pos
    scale_pos_weight = (neg / pos) if pos > 0 else 1.0

    clf = XGBClassifier(
        # device='cuda',
        n_estimators=400,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.8,
        objective='binary:logistic',
        scale_pos_weight=scale_pos_weight,
        random_state=int(random_state),
        n_jobs=1,
        eval_metric='logloss',
        tree_method='hist',
        # eval_metric='logloss',
        # tree_method='gpu_hist',      # For GPU training 
        # predictor='gpu_predictor',   # GPU prediction (optional)
    )

    clf.fit(X_train, y_train)
    return clf

## SIMULATOR

In [6]:
# Cell 6 - Simulation function

import pandas as pd
import numpy as np


def _daily_tail_cvar_return(daily_returns, alpha, lookback, min_history):
    """Historical CVaR (expected shortfall) of daily returns; negative = loss."""
    if len(daily_returns) < min_history:
        return None
    w = np.asarray(daily_returns[-lookback:], dtype=float)
    if len(w) < min_history:
        return None
    q = np.percentile(w, 100.0 * alpha)
    tail = w[w <= q]
    if tail.size == 0:
        return None
    return float(np.mean(tail))


def run_simulation(
    df_model,
    df_tf,
    X_test,
    y_pred,
    initial_equity=100_000.0,
    take_profit_pct=0.05,        # 5% TP
    stop_loss_pct=0.03,          # 3% SL
    max_open_trades=10,          # max concurrent positions
    pct_account_per_trade=1.0,   # 100% of available cash per trade
    commission_pct=0.001,        # 0.1% per side
    exit_mode=EXIT_MODE,           # "tp_sl" | "horizon" | "tp_sl_or_horizon"
    horizon_steps=None,          # bars to hold before time exit; required for horizon / combined
    cvar_loss_limit_pct_of_cvar=None,  # None = disabled; halt if day's loss >= (X/100)*|CVaR|
    cvar_alpha=0.05,
    cvar_lookback_days=60,
    cvar_min_history_days=20,
    cvar_halt_flatten=True,
):
    """
    Run a trading simulation on the test period using predicted buy signals.

    exit_mode
      * "tp_sl": exit only via take-profit / stop-loss (or forced flatten at end of test data).
      * "horizon": exit at the close of the bar when the trade has been open for `horizon_steps`
        bars (no TP/SL). Any position still open at the last bar closes at "EOD".
      * "tp_sl_or_horizon": TP/SL as above; if neither triggers within `horizon_steps` bars after
        entry, exit at that bar's close with reason "HORIZON".

    Daily CVaR stop (when cvar_loss_limit_pct_of_cvar is not None):
      Rolling CVaR of *completed* calendar-day returns on the strategy equity curve.
      If today's loss (from today's open equity to now) exceeds
          (cvar_loss_limit_pct_of_cvar / 100) * |CVaR|,
      set risk halt: no new trades; optionally flatten all positions at this bar's close ("CVAR_HALT").
    """
    if exit_mode not in ("tp_sl", "horizon", "tp_sl_or_horizon"):
        raise ValueError('exit_mode must be "tp_sl", "horizon", or "tp_sl_or_horizon"')
    if exit_mode in ("horizon", "tp_sl_or_horizon"):
        if horizon_steps is None or int(horizon_steps) < 1:
            raise ValueError("horizon_steps must be a positive integer for this exit_mode")
    horizon_steps = int(horizon_steps) if horizon_steps is not None else None

    # ==========================
    # PREP DATA
    # ==========================
    test_index = X_test.index.sort_values()

    # Close price at decision time t
    prices = df_model['Close'].loc[test_index]

    # Predicted buy signals aligned with test_index
    pred_buy = pd.Series(y_pred, index=test_index).astype(int)

    # Ensure df_tf has simple returns for buy & hold benchmark
    if 'returns' not in df_tf.columns:
        df_tf = df_tf.copy()
        df_tf['returns'] = df_tf['Close'].pct_change()

    # ==========================
    # SIMULATION LOOP
    # ==========================
    INITIAL_EQUITY = initial_equity
    TAKE_PROFIT_PCT = take_profit_pct
    STOP_LOSS_PCT = stop_loss_pct
    MAX_OPEN_TRADES = max_open_trades
    PCT_ACCOUNT_PER_TRADE = pct_account_per_trade
    COMMISSION_PCT = commission_pct

    equity_cash = INITIAL_EQUITY           # cash (no open positions)
    open_positions = []                    # list of open positions
    equity_curve = []                      # total portfolio value over time (cash + open positions)
    equity_index = []                      # timestamps
    trades = []                            # closed trade log
    total_commission = 0.0                 # track total commission paid

    last_total_equity = INITIAL_EQUITY
    current_calendar_day = None
    day_open_equity = INITIAL_EQUITY
    completed_daily_returns = []
    risk_halted = False
    cvar_halt_time = None
    cvar_halt_loss_limit_used = None

    def finalize_exit(pos, dt, exit_price, exit_reason):
        nonlocal equity_cash, total_commission
        entry_price = pos['entry_price']
        size = pos['size']
        entry_time = pos['entry_time']
        sell_value = size * exit_price
        sell_commission = sell_value * COMMISSION_PCT
        net_sell_value = sell_value - sell_commission
        pnl = net_sell_value - pos['cost_basis']
        ret = net_sell_value / pos['cost_basis'] - 1.0
        equity_cash += net_sell_value
        total_commission += sell_commission
        trades.append({
            'entry_time':  entry_time,
            'exit_time':   dt,
            'entry_price': entry_price,
            'exit_price':  exit_price,
            'size':        size,
            'pnl':         pnl,
            'return_pct':  ret * 100.0,
            'reason':      exit_reason,
        })

    for bar_i, (dt, price) in enumerate(prices.items()):
        ts = pd.Timestamp(dt)
        # Calendar day as in the data index (no timezone conversion — keeps exchange "day" stable).
        cal_day = ts.date()

        if current_calendar_day is None:
            current_calendar_day = cal_day
            day_open_equity = last_total_equity
        elif cal_day != current_calendar_day:
            dr = (last_total_equity - day_open_equity) / day_open_equity
            if np.isfinite(dr):
                completed_daily_returns.append(float(dr))
            current_calendar_day = cal_day
            day_open_equity = last_total_equity

        # --- 1) Update existing positions ---
        still_open = []
        for pos in open_positions:
            tp_price = pos['tp_price']
            sl_price = pos['sl_price']
            exit_reason = None
            exit_price = None

            use_tp_sl = exit_mode in ("tp_sl", "tp_sl_or_horizon")
            if use_tp_sl:
                if price >= tp_price:
                    exit_price = tp_price
                    exit_reason = 'TP'
                elif price <= sl_price:
                    exit_price = sl_price
                    exit_reason = 'SL'

            use_horizon = exit_mode in ("horizon", "tp_sl_or_horizon")
            if exit_reason is None and use_horizon:
                if bar_i - pos['entry_i'] >= horizon_steps:
                    exit_price = price
                    exit_reason = 'HORIZON'

            if exit_reason is not None:
                finalize_exit(pos, dt, exit_price, exit_reason)
            else:
                still_open.append(pos)

        open_positions = still_open

        # --- 2) Open new position if signal & capacity ---
        if (
            not risk_halted
            and pred_buy.loc[dt] == 1
            and len(open_positions) < MAX_OPEN_TRADES
            and equity_cash > 0
        ):
            trade_equity = equity_cash * PCT_ACCOUNT_PER_TRADE
            if trade_equity > 0:
                buy_commission = trade_equity * COMMISSION_PCT
                net_buy_value = trade_equity - buy_commission
                size = net_buy_value / price   # how many BTC we buy

                pos = {
                    'entry_time':  dt,
                    'entry_i':     bar_i,
                    'entry_price': price,
                    'size':        size,
                    'cost_basis':  trade_equity,  # includes buy commission
                    'tp_price':    price * (1 + TAKE_PROFIT_PCT),
                    'sl_price':    price * (1 - STOP_LOSS_PCT),
                }

                equity_cash -= trade_equity
                total_commission += buy_commission
                open_positions.append(pos)

        # --- 3) Mark-to-market portfolio value ---
        open_value = sum(pos['size'] * price for pos in open_positions)
        total_equity = equity_cash + open_value

        # --- 4) Daily CVaR loss limit (intraday) ---
        if (
            cvar_loss_limit_pct_of_cvar is not None
            and not risk_halted
            and day_open_equity > 0
        ):
            cvar_ret = _daily_tail_cvar_return(
                completed_daily_returns,
                alpha=cvar_alpha,
                lookback=cvar_lookback_days,
                min_history=cvar_min_history_days,
            )
            if cvar_ret is not None:
                tail_mag = abs(cvar_ret)
                if tail_mag > 0.0:
                    allowed_loss = (float(cvar_loss_limit_pct_of_cvar) / 100.0) * tail_mag
                    day_ret = total_equity / day_open_equity - 1.0
                    if day_ret < 0.0 and (-day_ret) >= allowed_loss:
                        risk_halted = True
                        cvar_halt_time = dt
                        cvar_halt_loss_limit_used = allowed_loss
                        if cvar_halt_flatten:
                            for pos in list(open_positions):
                                finalize_exit(pos, dt, price, 'CVAR_HALT')
                            open_positions = []
                            total_equity = equity_cash

        equity_curve.append(total_equity)
        equity_index.append(dt)
        last_total_equity = total_equity

    # ==========================
    # CLOSE ANY REMAINING OPEN POSITIONS AT LAST PRICE
    # ==========================
    if len(open_positions) > 0:
        last_dt = prices.index[-1]
        last_price = prices.iloc[-1]
        for pos in open_positions:
            finalize_exit(pos, last_dt, last_price, 'EOD')

        open_positions = []

    # Final portfolio value (all in cash now)
    final_equity = equity_cash
    total_return_pct = (final_equity / INITIAL_EQUITY - 1.0) * 100.0

    # Return without commission (hypothetical)
    final_equity_no_commission = final_equity + total_commission
    total_return_no_commission_pct = (final_equity_no_commission / INITIAL_EQUITY - 1.0) * 100.0
    commission_impact_pct = total_return_no_commission_pct - total_return_pct

    equity_series = pd.Series(equity_curve, index=equity_index)
    trades_df = pd.DataFrame(trades).sort_values('entry_time')

    # ==========================
    # BUY & HOLD BENCHMARK
    # ==========================
    bh_returns = df_tf.loc[test_index, 'returns'].dropna()
    bh_cumulative_return = (1 + bh_returns).prod() - 1.0
    bh_total_return_pct = bh_cumulative_return * 100.0

    # ==========================
    # CALCULATE CAGR AND ANNUAL RETURNS
    # ==========================
    start_date = test_index[0]
    end_date = test_index[-1]
    years = max((end_date - start_date).days / 365.25, 1e-6)  # avoid div by zero

    # Model CAGR
    model_cagr = (final_equity / INITIAL_EQUITY) ** (1 / years) - 1.0
    model_cagr_pct = model_cagr * 100.0

    # Model CAGR without commission
    model_cagr_no_commission = (final_equity_no_commission / INITIAL_EQUITY) ** (1 / years) - 1.0
    model_cagr_no_commission_pct = model_cagr_no_commission * 100.0

    # Bitcoin Buy & Hold CAGR
    bh_final_value = INITIAL_EQUITY * (1 + bh_cumulative_return)
    bh_cagr = (bh_final_value / INITIAL_EQUITY) ** (1 / years) - 1.0
    bh_cagr_pct = bh_cagr * 100.0

    # ==========================
    # SUMMARY
    # ==========================
    num_trades = len(trades_df)
    wins = (trades_df['pnl'] > 0).sum()
    losses = (trades_df['pnl'] < 0).sum()
    win_rate = (wins / num_trades * 100.0) if num_trades > 0 else 0.0

    summary = {
        "initial_equity": INITIAL_EQUITY,
        "final_equity": final_equity,
        "total_return_pct": total_return_pct,
        "num_trades": num_trades,
        "wins": wins,
        "losses": losses,
        "win_rate": win_rate,
        "model_cagr_pct": model_cagr_pct,
        "total_commission": total_commission,
        "total_return_no_commission_pct": total_return_no_commission_pct,
        "model_cagr_no_commission_pct": model_cagr_no_commission_pct,
        "commission_impact_pct": commission_impact_pct,
        "bh_total_return_pct": bh_total_return_pct,
        "bh_cagr_pct": bh_cagr_pct,
        "start_date": start_date,
        "end_date": end_date,
        "years": years,
        "exit_mode": exit_mode,
        "horizon_steps": horizon_steps,
        "cvar_halted": cvar_halt_time is not None,
        "cvar_halt_time": cvar_halt_time,
        "cvar_loss_limit_pct_of_cvar": cvar_loss_limit_pct_of_cvar,
        "cvar_halt_loss_fraction_cap": cvar_halt_loss_limit_used,
    }

    return equity_series, trades_df, summary


## EVALUATE

In [7]:
# Cell 7 - Evaluate a full configuration
#
# To replay a known winner directly (no random search), call evaluate_config with exact params, e.g.:
#   evaluate_config(
#       resample_rule="4h", horizon_steps=6, target_simple_return=0.003472,
#       take_profit_pct=0.070650, stop_loss_pct=0.007443, max_open_trades=4,
#       pct_account_per_trade=0.439777, threshold=0.313592, verbose=True,
#   )

def evaluate_config(
    resample_rule,
    horizon_steps,
    target_simple_return,
    take_profit_pct,
    stop_loss_pct,
    max_open_trades,
    pct_account_per_trade,
    threshold,
    verbose=False,
    random_state=None,
    exit_mode=None,
    cvar_loss_limit_pct_of_cvar=None,
    cvar_alpha=None,
    cvar_lookback_days=None,
    cvar_min_history_days=None,
    cvar_halt_flatten=None,
):
    _exit_mode = EXIT_MODE if exit_mode is None else exit_mode
    _cvar_lim = CVAR_LOSS_LIMIT_PCT_OF_CVAR if cvar_loss_limit_pct_of_cvar is None else cvar_loss_limit_pct_of_cvar
    _cvar_a = CVAR_ALPHA if cvar_alpha is None else cvar_alpha
    _cvar_lb = CVAR_LOOKBACK_DAYS if cvar_lookback_days is None else cvar_lookback_days
    _cvar_mh = CVAR_MIN_HISTORY_DAYS if cvar_min_history_days is None else cvar_min_history_days
    _cvar_flt = CVAR_HALT_FLATTEN if cvar_halt_flatten is None else cvar_halt_flatten
    _model_seed = XGBOOST_RANDOM_STATE if random_state is None else int(random_state)

    # 1) Build dataset for this timeframe / horizon / target
    df_tf, df_model = build_dataset(
        resample_rule=resample_rule,
        horizon_steps=horizon_steps,
        target_simple_return=target_simple_return
    )

    # Feature columns
    # Standard - Same Equity Curve (2025-06-25): must match export_simulation_equity_curve.py + api_liveScript.py
    feature_cols = FEATURE_COLS

    # Guard: if too few rows, bail out
    if len(df_model) < 500:
        return None  # not enough data for this config

    # 2) Train/test split
    X_train, X_test, y_train, y_test, train_start, train_end, test_start, test_end = split_train_test(
        df_model, feature_cols
    )

    if len(X_train) == 0 or len(X_test) == 0:
        return None

    # Guard: we need some positive samples to train
    if y_train.sum() == 0:
        return None

    # 3) Train classifier (fixed hyperparams)
    clf = train_classifier(X_train, y_train, random_state=_model_seed)

    # 4) Predict probabilities and apply threshold
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)

    # If no trades triggered, skip this config
    if y_pred.sum() == 0:
        return None

    # 5) Run simulation
    equity_series, trades_df, summary = run_simulation(
        df_model=df_model,
        df_tf=df_tf,
        X_test=X_test,
        y_pred=y_pred,
        initial_equity=INITIAL_EQUITY,
        take_profit_pct=take_profit_pct,
        stop_loss_pct=stop_loss_pct,
        max_open_trades=max_open_trades,
        pct_account_per_trade=pct_account_per_trade,
        commission_pct=COMMISSION_PCT,
        exit_mode=_exit_mode,
        horizon_steps=horizon_steps,
        cvar_loss_limit_pct_of_cvar=_cvar_lim,
        cvar_alpha=_cvar_a,
        cvar_lookback_days=_cvar_lb,
        cvar_min_history_days=_cvar_mh,
        cvar_halt_flatten=_cvar_flt,
    )

    if verbose:
        print("Config:")
        print(f"  RESAMPLE_RULE = {resample_rule}")
        print(f"  HORIZON_STEPS = {horizon_steps}")
        print(f"  TARGET_SIMPLE_RETURN = {target_simple_return:.4f}")
        print(f"  EXIT_MODE = {_exit_mode}")
        print(f"  CVaR halt: limit%of|CVaR|={_cvar_lim}, alpha={_cvar_a}, lookback={_cvar_lb}, min_hist={_cvar_mh}, flatten={_cvar_flt}")
        print(f"  TP = {take_profit_pct:.3f}, SL = {stop_loss_pct:.3f}, threshold = {threshold:.3f}")
        print(f"  final_equity = {summary['final_equity']:.2f}, total_return = {summary['total_return_pct']:.2f}%")
        print(f"  num_trades = {summary['num_trades']}, win_rate = {summary['win_rate']:.2f}%")

    # Use final_equity as optimization score
    result = {
        "resample_rule": resample_rule,
        "horizon_steps": horizon_steps,
        "target_simple_return": target_simple_return,
        "take_profit_pct": take_profit_pct,
        "stop_loss_pct": stop_loss_pct,
        "max_open_trades": max_open_trades,
        "pct_account_per_trade": pct_account_per_trade,
        "threshold": threshold,
        "exit_mode": _exit_mode,
        "cvar_loss_limit_pct_of_cvar": _cvar_lim,
        "cvar_halted": summary.get("cvar_halted"),
        "final_equity": summary['final_equity'],
        "total_return_pct": summary['total_return_pct'],
        "model_cagr_pct": summary['model_cagr_pct'],
        "num_trades": summary['num_trades'],
        "win_rate": summary['win_rate'],
        "bh_total_return_pct": summary['bh_total_return_pct'],
        "bh_cagr_pct": summary['bh_cagr_pct'],
    }
    return result


## RANDOM SEARCH

Reproducibility requires **all** of these to match between runs:

- `SEARCH_SEED` and `N_TRIALS` (cell 1)
- `RESAMPLE_OPTIONS` / `grid_fingerprint` (derived from `BASE_BAR_MINUTES`)
- `DATA_PATH`, train/test dates, `EXIT_MODE`

Trials are pre-generated by `generate_search_trials()` before the loop. Re-run a single trial: `replay_search_trial(797)`.

In [8]:
import xgboost as xgb
print(xgb.__version__)

1.7.6


In [9]:
import numpy as np
import time
import pandas as pd

# Uses SEARCH_SEED + N_TRIALS from cell 1. Same seed + grid -> identical SEARCH_TRIALS every run.
exit_mode_run = EXIT_MODE
cvar_limit_run = CVAR_LOSS_LIMIT_PCT_OF_CVAR

RESAMPLE_OPTIONS = resample_options_for_base(BASE_BAR_MINUTES)
GRID_FP = search_grid_fingerprint(RESAMPLE_OPTIONS)
SEARCH_TRIALS = generate_search_trials(N_TRIALS, SEARCH_SEED, RESAMPLE_OPTIONS)

print(
    f"Search: seed={SEARCH_SEED}, trials={N_TRIALS}, grid={GRID_FP}, "
    f"rules={list(RESAMPLE_OPTIONS.keys())}"
)
print("Trial 0 sample:", {k: SEARCH_TRIALS[0][k] for k in SEARCH_TRIALS[0] if k != "grid_fingerprint"})

results = []
start_time = time.time()
best_ret_pct = -np.inf
best_trial = None
estimated_time_printed = False

for t in SEARCH_TRIALS:
    i = t["trial"]
    res = evaluate_config(
        resample_rule=t["resample_rule"],
        horizon_steps=t["horizon_steps"],
        target_simple_return=t["target_simple_return"],
        take_profit_pct=t["take_profit_pct"],
        stop_loss_pct=t["stop_loss_pct"],
        max_open_trades=t["max_open_trades"],
        pct_account_per_trade=t["pct_account_per_trade"],
        threshold=t["threshold"],
        random_state=t["model_random_state"],
        verbose=False,
        exit_mode=exit_mode_run,
        cvar_loss_limit_pct_of_cvar=cvar_limit_run,
    )

    if res is not None:
        res["trial"] = i
        res["search_seed"] = SEARCH_SEED
        res["grid_fingerprint"] = GRID_FP
        res["model_random_state"] = t["model_random_state"]
        results.append(res)

        if res["total_return_pct"] > best_ret_pct:
            best_ret_pct = res["total_return_pct"]
            best_trial = i

        print(
            f"Trial {i:03d}: final_equity={res['final_equity']:.2f}, "
            f"ret={res['total_return_pct']:.2f}%, trades={res['num_trades']}, "
            f"current_best={best_ret_pct:.2f}%"
        )

        if not estimated_time_printed:
            elapsed = time.time() - start_time
            est_total = elapsed * N_TRIALS / (i + 1)
            est_minutes = est_total / 60.0
            print(
                f"Estimated total time: {est_total:.2f} seconds "
                f"(~{est_minutes:.2f} minutes) for {N_TRIALS} trials"
            )
            estimated_time_printed = True
    else:
        print(f"Trial {i:03d}: skipped (no trades or bad config)")

results_df = pd.DataFrame(results)

print("\nTop 10 configs by final equity:")
print(results_df.sort_values("final_equity", ascending=False).head(10))

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution completed in: {execution_time:.2f} seconds")
print(f"Best trial: {best_trial}, best_return={best_ret_pct:.2f}%")


Trial 000: final_equity=126326.91, ret=26.33%, trades=290, current_best=26.33%
Estimated total time: 353.41 seconds (~5.89 minutes) for 800 trials
Trial 001: final_equity=179091.49, ret=79.09%, trades=1665, current_best=79.09%
Trial 002: final_equity=94157.13, ret=-5.84%, trades=328, current_best=79.09%
Trial 003: final_equity=97905.67, ret=-2.09%, trades=9, current_best=79.09%
Trial 004: final_equity=7716.88, ret=-92.28%, trades=4067, current_best=79.09%
Trial 005: final_equity=137663.39, ret=37.66%, trades=585, current_best=79.09%
Trial 006: final_equity=95909.72, ret=-4.09%, trades=27, current_best=79.09%
Trial 007: final_equity=278588.68, ret=178.59%, trades=1890, current_best=178.59%
Trial 008: final_equity=111089.27, ret=11.09%, trades=403, current_best=178.59%
Trial 009: final_equity=99161.95, ret=-0.84%, trades=1, current_best=178.59%
Trial 010: final_equity=102300.73, ret=2.30%, trades=84, current_best=178.59%
Trial 011: final_equity=72249.84, ret=-27.75%, trades=895, current_

In [10]:
print("\nTop 10 configs by final equity:")
results_df.sort_values('final_equity', ascending=False).head(10)


Top 10 configs by final equity:


,resample_rule,horizon_steps,target_simple_return,take_profit_pct,stop_loss_pct,max_open_trades,pct_account_per_trade,threshold,exit_mode,cvar_loss_limit_pct_of_cvar,cvar_halted,final_equity,total_return_pct,model_cagr_pct,num_trades,win_rate,bh_total_return_pct,bh_cagr_pct,trial
745,4h,6,0.003472,0.070650,0.007443,4,0.439777,0.313592,tp_sl_or_horizon,None,False,4.638632e+06,4538.631988,133.577792,6890,38.098694,13.785140,2.896421,797
618,1h,48,0.017960,0.094575,0.008043,9,0.428113,0.342187,tp_sl_or_horizon,None,False,2.110932e+06,2010.932099,96.825605,4099,20.809954,19.220260,3.980643,666
684,1h,8,0.003647,0.050192,0.005273,8,0.243774,0.349152,tp_sl_or_horizon,None,False,2.021182e+06,1921.182474,94.385422,8063,33.101823,13.321421,2.803559,734
134,4h,12,0.020736,0.090651,0.009518,4,0.430915,0.407695,tp_sl_or_horizon,None,False,1.382718e+06,1282.718371,78.798561,3270,35.871560,14.033208,2.947788,144
592,4h,6,0.003930,0.091693,0.008199,6,0.252916,0.424424,tp_sl_or_horizon,None,False,1.217136e+06,1117.136453,73.765591,6604,39.127801,13.785140,2.896421,639
611,1D,7,0.045498,0.073001,0.005331,10,0.326983,0.377854,tp_sl_or_horizon,None,False,7.510942e+05,651.094236,56.429255,984,29.065041,17.456698,3.634890,658
598,1h,24,0.044130,0.056723,0.010287,9,0.448075,0.405003,tp_sl_or_horizon,None,False,5.199484e+05,419.948442,44.073359,3793,29.448985,15.730603,3.288898,645
388,1h,48,0.056003,0.080449,0.014282,8,0.273662,0.319762,tp_sl_or_horizon,None,False,4.723527e+05,372.352707,41.160028,2574,28.399378,19.220260,3.980643,422
199,1h,48,0.061393,0.050051,0.007840,6,0.485939,0.486393,tp_sl_or_horizon,None,False,4.188306e+05,318.830572,37.440652,1907,24.855794,19.220260,3.980643,215
650,4h,12,0.023790,0.042950,0.007190,2,0.495346,0.596083,tp_sl_or_horizon,None,False,3.949630e+05,294.962961,35.511422,1336,34.056886,14.033208,2.947788,698


## Standard - Same Equity Curve — export production joblib

Run the next cell after loading data (2025-06-25). Writes Models/BTCUSDT4h1307.joblib for live + export parity.


In [27]:
# ---------------------------------------------------------------------------
# Standard - Same Equity Curve — production model + equity-curve inputs (2025-06-25)
#
# Trains the classifier live bot / export script expect:
#   - Same FEATURE_COLS as features.py
#   - Same hyperparams as export_simulation_equity_curve.py BacktestConfig
#   - Output: Models/BTCUSDT4h1307.joblib → set MODEL_PATH in .env
# ---------------------------------------------------------------------------

import joblib
from pathlib import Path

Path("results").mkdir(exist_ok=True)

# Standard - Same Equity Curve (2025-06-25): aligned with export_simulation_equity_curve.py BacktestConfig
PROD_RESAMPLE_RULE = "4h"
PROD_HORIZON_STEPS = 6
PROD_TARGET_SIMPLE_RETURN = 0.003472

df_tf, df_model = build_dataset(
    resample_rule=PROD_RESAMPLE_RULE,
    horizon_steps=PROD_HORIZON_STEPS,
    target_simple_return=PROD_TARGET_SIMPLE_RETURN,
)

X_train, X_test, y_train, y_test, train_start, train_end, test_start, test_end = split_train_test(
    df_model, FEATURE_COLS
)

print(f"=== Standard - Same Equity Curve — PRODUCTION MODEL (2025-06-25) ===")
print(f"Train: {train_start} -> {train_end}  ({len(X_train):,} rows)")
print(f"Test:  {test_start} -> {test_end}  ({len(X_test):,} rows)")
print(f"Positives: {int(y_train.sum()):,} / {len(y_train):,}")
print(f"Features: {FEATURE_COLS}")

prod_clf = train_classifier(X_train, y_train)
joblib.dump(prod_clf, MODEL_OUTPUT_PATH)

print(f"\nSaved production model: {Path(MODEL_OUTPUT_PATH).resolve()}")
print("Next steps (2025-06-25):")
print("  1. MODEL_PATH=Models/BTCUSDT4h1307.joblib in .env")
print("  2. THRESHOLD=0.4214, TAKE_PROFIT=0.0455, STOP_LOSS=0.0051, PCT_ACCOUNT_PER_TRADE=0.7888")
print("  3. Re-run export_simulation_equity_curve.py with same joblib for equity-curve compare")

# Optional: best random-search model (also uses FEATURE_COLS after 2025-06-25 unification)
if "results_df" in globals() and len(results_df) > 0:
    best_idx = results_df["total_return_pct"].idxmax()
    best_config = results_df.loc[best_idx]

    df_tf_b, df_model_b = build_dataset(
        resample_rule=best_config["resample_rule"],
        horizon_steps=int(best_config["horizon_steps"]),
        target_simple_return=best_config["target_simple_return"],
    )
    X_train_b, _, y_train_b, _, *_ = split_train_test(df_model_b, FEATURE_COLS)
    best_clf = train_classifier(X_train_b, y_train_b)

    search_filename = (
        Path("results")
        / f"ModelId{best_idx}_{best_config['total_return_pct']:.2f}%_"
        f"{best_config['resample_rule']}_{int(best_config['horizon_steps'])}.joblib"
    )
    joblib.dump(best_clf, search_filename)
    print(f"\nSaved best search model: {search_filename}")
    print(f"  Total return pct: {best_config['total_return_pct']:.2f}%")


Saved best model to: results\ModelId745_4538.63%_4h_6.joblib
Best model performance:
  Total return pct: 4538.63%
  Resample rule: 4h
  Horizon steps: 6


In [11]:
from datetime import datetime
from pathlib import Path

# Create folder
Path("results").mkdir(exist_ok=True)

# Sort from BEST to WORST
sorted_df = results_df.sort_values(
    by="final_equity",
    ascending=False
)

# Timestamped filename
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

filename = f"results/V2_Bot_Results_{timestamp}_{EXIT_MODE}.xlsx"

# Export to Excel
sorted_df.to_excel(filename, index=False)

print(f"Saved: {filename}")

Saved: results/V2_Bot_Results_2026-06-25_17-41-44_tp_sl_or_horizon.xlsx


In [12]:
# Extract the top configuration by final equity
top_config = results_df.sort_values('final_equity', ascending=False).iloc[0]

# Rebuild the dataset and run the simulation for the top configuration
df_tf, df_model = build_dataset(
    resample_rule=top_config['resample_rule'],
    horizon_steps=top_config['horizon_steps'],
    target_simple_return=top_config['target_simple_return']
)

# Standard - Same Equity Curve (2025-06-25): FEATURE_COLS shared with export + live
feature_cols = FEATURE_COLS

X_train, X_test, y_train, y_test, train_start, train_end, test_start, test_end = split_train_test(
    df_model, feature_cols
)

clf = train_classifier(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= top_config['threshold']).astype(int)

_, trades_df, _ = run_simulation(
    df_model=df_model,
    df_tf=df_tf,
    X_test=X_test,
    y_pred=y_pred,
    initial_equity=INITIAL_EQUITY,
    take_profit_pct=top_config['take_profit_pct'],
    stop_loss_pct=top_config['stop_loss_pct'],
    max_open_trades=top_config['max_open_trades'],
    pct_account_per_trade=top_config['pct_account_per_trade'],
    commission_pct=COMMISSION_PCT,
    exit_mode=top_config.get('exit_mode', EXIT_MODE),
    horizon_steps=int(top_config['horizon_steps']),
    cvar_loss_limit_pct_of_cvar=top_config.get('cvar_loss_limit_pct_of_cvar', CVAR_LOSS_LIMIT_PCT_OF_CVAR),
    cvar_alpha=CVAR_ALPHA,
    cvar_lookback_days=CVAR_LOOKBACK_DAYS,
    cvar_min_history_days=CVAR_MIN_HISTORY_DAYS,
    cvar_halt_flatten=CVAR_HALT_FLATTEN,
)

# Calculate statistics
trades_df['entry_date'] = trades_df['entry_time'].dt.date
daily_trades = trades_df.groupby('entry_date').size()
average_trades_per_day = daily_trades.mean()

weekly_trades = trades_df['entry_date'].apply(lambda x: x.isocalendar()[1]).value_counts()
average_trades_per_week = weekly_trades.mean()

monthly_trades = trades_df['entry_date'].apply(lambda x: x.month).value_counts()
average_trades_per_month = monthly_trades.mean()

print(f"Average trades per day: {average_trades_per_day:.2f}")
print(f"Average trades per week: {average_trades_per_week:.2f}")
print(f"Average trades per month: {average_trades_per_month:.2f}")

Average trades per day: 4.21
Average trades per week: 132.50
Average trades per month: 574.17


In [13]:
import plotly.graph_objects as go

# Standard - Same Equity Curve (2025-06-25): top-N replays use same FEATURE_COLS as production export

# Sort and take top 3 configs
top3 = results_df.sort_values('final_equity', ascending=False).head(3).reset_index(drop=True)

def rebuild_and_simulate(row):
    """
    Given one row from results_df (a config), rebuild the full pipeline:
    - build dataset
    - split train/test
    - train classifier
    - get y_pred with its threshold
    - run simulation
    Return equity_series, trades_df, summary, and df_model/df_tf used.
    """
    resample_rule        = row['resample_rule']
    horizon_steps        = int(row['horizon_steps'])
    target_simple_return = row['target_simple_return']
    take_profit_pct      = row['take_profit_pct']
    stop_loss_pct        = row['stop_loss_pct']
    max_open_trades      = int(row['max_open_trades'])
    pct_account_per_trade = row['pct_account_per_trade']
    threshold            = row['threshold']

    # 1) Build dataset
    df_tf, df_model = build_dataset(
        resample_rule=resample_rule,
        horizon_steps=horizon_steps,
        target_simple_return=target_simple_return
    )

    # Standard - Same Equity Curve (2025-06-25): FEATURE_COLS shared with export + live
feature_cols = FEATURE_COLS

    X_train, X_test, y_train, y_test, *_ = split_train_test(
        df_model, feature_cols
    )

    clf = train_classifier(X_train, y_train)

    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)

    equity_series, trades_df, summary = run_simulation(
        df_model=df_model,
        df_tf=df_tf,
        X_test=X_test,
        y_pred=y_pred,
        initial_equity=INITIAL_EQUITY,
        take_profit_pct=take_profit_pct,
        stop_loss_pct=stop_loss_pct,
        max_open_trades=max_open_trades,
        pct_account_per_trade=pct_account_per_trade,
        commission_pct=COMMISSION_PCT,
        exit_mode=row.get('exit_mode', EXIT_MODE),
        horizon_steps=horizon_steps,
        cvar_loss_limit_pct_of_cvar=row.get('cvar_loss_limit_pct_of_cvar', CVAR_LOSS_LIMIT_PCT_OF_CVAR),
        cvar_alpha=CVAR_ALPHA,
        cvar_lookback_days=CVAR_LOOKBACK_DAYS,
        cvar_min_history_days=CVAR_MIN_HISTORY_DAYS,
        cvar_halt_flatten=CVAR_HALT_FLATTEN,
    )

    return equity_series, trades_df, summary, df_model, df_tf


for idx, row in top3.iterrows():
    print(f"\n=== Strategy #{idx+1} ===")
    print(f"Resample: {row['resample_rule']}, Horizon: {int(row['horizon_steps'])} bars")
    print(f"Target simple return: {row['target_simple_return']:.4f}")
    print(f"TP: {row['take_profit_pct']:.3f}, SL: {row['stop_loss_pct']:.3f}, "
          f"Threshold: {row['threshold']:.3f}")
    print(f"Max open trades: {int(row['max_open_trades'])}, "
          f"% per trade: {row['pct_account_per_trade']:.3f}")
    print(f"Final equity (search run): {row['final_equity']:.2f}, "
          f"Total return: {row['total_return_pct']:.2f}%")

    equity_series, trades_df, summary, df_model_used, df_tf_used = rebuild_and_simulate(row)

    print(f"Rebuilt final equity: {summary['final_equity']:.2f}, "
          f"Total return: {summary['total_return_pct']:.2f}%, "
          f"Trades: {summary['num_trades']}, Win rate: {summary['win_rate']:.2f}%")

    # --- Plot equity curve with trade markers (Plotly) ---
    fig = go.Figure()

    # Equity line
    fig.add_trace(
        go.Scatter(
            x=equity_series.index,
            y=equity_series.values,
            mode="lines",
            name="Equity"
        )
    )

    # Trade entries & exits
    if len(trades_df) > 0:
        # Ensure they align with equity_series index
        entry_mask = trades_df["entry_time"].isin(equity_series.index)
        exit_mask  = trades_df["exit_time"].isin(equity_series.index)

        entry_times = trades_df.loc[entry_mask, "entry_time"]
        exit_times  = trades_df.loc[exit_mask, "exit_time"]

        # Entries
        if len(entry_times) > 0:
            fig.add_trace(
                go.Scatter(
                    x=entry_times,
                    y=equity_series.loc[entry_times],
                    mode="markers",
                    name="Entry",
                    marker=dict(symbol="triangle-up", size=10, opacity=0.8)
                )
            )

        # Exits
        if len(exit_times) > 0:
            fig.add_trace(
                go.Scatter(
                    x=exit_times,
                    y=equity_series.loc[exit_times],
                    mode="markers",
                    name="Exit",
                    marker=dict(symbol="triangle-down", size=10, opacity=0.8)
                )
            )

    fig.update_layout(
        title=(
            f"Strategy #{idx+1} — Equity Curve<br>"
            f"{row['resample_rule']}, H={int(row['horizon_steps'])}, "
            f"TP={row['take_profit_pct']:.3f}, SL={row['stop_loss_pct']:.3f}, "
            f"thr={row['threshold']:.3f}"
        ),
        xaxis_title="Time",
        yaxis_title="Equity",
        legend=dict(x=0.01, y=0.99),
        hovermode="x unified",
    )
    fig.update_yaxes(type="log")
    fig.show()



=== Strategy #1 ===
Resample: 4h, Horizon: 6 bars
Target simple return: 0.0035
TP: 0.071, SL: 0.007, Threshold: 0.314
Max open trades: 4, % per trade: 0.440
Final equity (search run): 4638631.99, Total return: 4538.63%
Rebuilt final equity: 4638631.99, Total return: 4538.63%, Trades: 6890, Win rate: 38.10%



=== Strategy #2 ===
Resample: 1h, Horizon: 48 bars
Target simple return: 0.0180
TP: 0.095, SL: 0.008, Threshold: 0.342
Max open trades: 9, % per trade: 0.428
Final equity (search run): 2110932.10, Total return: 2010.93%
Rebuilt final equity: 2110932.10, Total return: 2010.93%, Trades: 4099, Win rate: 20.81%



=== Strategy #3 ===
Resample: 1h, Horizon: 8 bars
Target simple return: 0.0036
TP: 0.050, SL: 0.005, Threshold: 0.349
Max open trades: 8, % per trade: 0.244
Final equity (search run): 2021182.47, Total return: 1921.18%
Rebuilt final equity: 2021182.47, Total return: 1921.18%, Trades: 8063, Win rate: 33.10%


In [14]:
# import pickle

# # Save the trained classifier of the top configuration
# with open("best_model.pkl", "wb") as f:
#     pickle.dump(clf, f)

# print("Best model saved as 'best_model.pkl'")